# 02 · Industry payments and brand-name prescribing

Builds up the headline "industry-paid prescribers wrote X% more brand-name
fills than peers": crude gap -> peer-adjusted O/E -> regression with case-mix
controls -> dose-response -> drug-matched payments. Definitions:
`docs/methodology.md` section 3.

In [ ]:
import numpy as np
import pandas as pd

from _setup import OCFG, WH, banner, q, table
from partd_risk.modeling.brand_effect import fit_brand_effect, fit_tier_effects

banner()

## Crude vs. peer-adjusted

In [ ]:
comparison = WH.read("reporting", "rpt_brand_comparison").set_index("payment_group")
comparison[
    ["n_prescribers", "crude_brand_share", "brand_oe_ratio",
     "pct_more_brand_fills_crude", "pct_more_brand_fills_peer_adjusted",
     "pct_more_multisource_brand_fills_peer_adjusted"]
].T  # fmt: skip

The crude gap mixes specialty differences into the comparison; the
peer-adjusted gap compares each prescriber with unpaid peers in the same
specialty and state. The difference between the two rows above shows how
much of the crude gap is specialty/geography mix.

## Does the name-based brand rule agree with CMS's classification?

In [ ]:
q(
    f"""
    select
        corr(brand_fill_share, cms_brand_claim_share) as correlation,
        avg(abs(brand_fill_share - cms_brand_claim_share)) as mean_abs_diff,
        count(*) as n
    from {table('marts', 'fct_prescribers')}
    where in_cohort and cms_brand_claim_share is not null
    """
)

## Regression check (Poisson GLM, peer-expected offset, case-mix controls)

In [ ]:
brand = WH.read("marts", "mart_brand_peer_comparison")
features = WH.read("marts", "mart_outlier_features", ["npi", *OCFG.controls])
brand = brand.merge(features, on="npi", how="left")
effect, summary = fit_brand_effect(brand, list(OCFG.controls))
pd.Series(effect.as_dict())

In [ ]:
print(summary)

## Dose-response

In [ ]:
tiers = WH.read("reporting", "rpt_brand_by_payment_tier").sort_values(["dimension", "dimension_value"])
tiers[["dimension", "dimension_value", "n_prescribers", "brand_oe_ratio", "pct_vs_expected"]]

In [ ]:
fit_tier_effects(brand, list(OCFG.controls))

## Heterogeneity by specialty

In [ ]:
by_spec = (
    brand.groupby(["prescriber_specialty", "is_industry_paid"])[["brand_fills", "expected_brand_fills"]]
    .sum()
    .assign(oe=lambda d: d["brand_fills"] / d["expected_brand_fills"])
    ["oe"]
    .unstack()
    .rename(columns={False: "unpaid_oe", True: "paid_oe"})
)
by_spec["pct_more"] = 100 * (by_spec["paid_oe"] / by_spec["unpaid_oe"] - 1)
n = brand.groupby("prescriber_specialty").size().rename("n")
by_spec.join(n).query("n >= 200").sort_values("pct_more", ascending=False).head(20)

## Drug-matched: paid in connection with brand B -> more brand B?

In [ ]:
matched = WH.read("marts", "mart_drug_matched_prescribing")
matched["log_oe"] = np.log(matched["oe_ratio"])
print(f"{len(matched)} brands; median O/E = {matched['oe_ratio'].median():.2f}")
matched.sort_values("n_paid_prescribers", ascending=False).head(20)[
    ["brand_name", "n_paid_prescribers", "payment_usd", "observed_brand_fills",
     "expected_brand_fills", "oe_ratio"]
]  # fmt: skip